# reticolo — CUDA backend build & test on Colab

Compiles `reticolo::cuda` and runs the Phase 0 self-test on a real GPU — the toolchain check we can't do locally (no nvcc on the dev Mac).

**Before running:** `Runtime → Change runtime type → Hardware accelerator → GPU`.

Open in Colab:
https://colab.research.google.com/github/olmo-francesconi/reticolo/blob/feat/cuda-extension/tools/colab_cuda_build.ipynb

## 1. GPU & toolchain check

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv
!nvcc --version

## 2. Recent CMake + Ninja

reticolo needs CMake ≥ 3.25 (and ≥ 3.24 for `CUDA_ARCHITECTURES=native`). Install both from pip so we don't depend on the image's apt versions.

In [ ]:
!pip install -q "cmake>=3.25" ninja
!cmake --version | head -1
!ninja --version

## 3. Clone the CUDA branch

In [ ]:
REPO   = "https://github.com/olmo-francesconi/reticolo.git"
BRANCH = "feat/cuda-extension"

!rm -rf reticolo
!git clone --depth 1 --branch {BRANCH} {REPO}
%cd reticolo
!git log --oneline -1

## 4. Build & test

All the work is in `tools/cuda_build_test.sh` — the same headless script a console runner (Kaggle CLI, a GPU GitHub Actions runner, papermill) would call. `CUDA_ARCHITECTURES=native` targets exactly the GPU Colab handed us (T4=75, L4=89, A100=80, …); IO / apps / examples are off so no HDF5 is needed for the Phase 0 gate.

In [ ]:
!bash tools/cuda_build_test.sh

## Notes

- **Full (CI-faithful) build:** `apt-get install -y libhdf5-dev`, then `cmake --preset cuda-linux -DCMAKE_CUDA_ARCHITECTURES=native && cmake --build build/cuda-linux && ctest --test-dir build/cuda-linux`.
- **Later phases:** the device reversibility / force-vs-FD / integrator-order tests land in Phase 2+ and run through the same script (`RETICOLO_TEST_FILTER=hmc tools/cuda_build_test.sh`).
- If the clone fails with *Remote branch not found*, the `feat/cuda-extension` branch hasn't been pushed to `origin` yet.